### HW 6

In [20]:
import os
import pyspark
from pyspark.sql import SparkSession

In [21]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

#### Q1.  Install Spark and PySpark.

In [22]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


Answer: `Spark version: 4.1.1`

---

#### Q2. What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [23]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

df.repartition(4).write.parquet("output/yellow_tripdata_2025-11", mode="overwrite")

In [24]:
folder = "output/yellow_tripdata_2025-11"
sizes = [
    os.path.getsize(os.path.join(folder, f))
    for f in os.listdir(folder)
    if f.endswith(".parquet")
]

avg_mb = (sum(sizes) / len(sizes)) / (1024 * 1024)
print(f"Average size: {avg_mb:.2f} MB")

Average size: 24.41 MB


Answer: `25MB`

---

#### Q3. How many taxi trips were there on the 15th of November?

In [26]:
from pyspark.sql.functions import dayofmonth, col

df.filter(dayofmonth("tpep_pickup_datetime") == 15).count()

162604

Answer: `162,604`

---

#### Q4. What is the length of the longest trip in the dataset in hours?

In [28]:
from pyspark.sql.functions import col, max, unix_timestamp

df.withColumn(
    "trip_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
).select(max("trip_hours")).show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



Answer: `90.6`

---

#### Q5. In Spark's UI, which shows the application's dashboard runs on which local port?

Answer: `4040`

---

#### Q6. Least frequent pickup location zone?

In [29]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
df_zones.createOrReplaceTempView("zones")

In [31]:
df.createOrReplaceTempView("trips")

spark.sql("""
    SELECT z.Zone, COUNT(*) as cnt
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY cnt ASC
    LIMIT 1
""").show(truncate=False)

+---------------------------------------------+---+
|Zone                                         |cnt|
+---------------------------------------------+---+
|Governor's Island/Ellis Island/Liberty Island|1  |
+---------------------------------------------+---+



Answer: `Governor's Island/Ellis Island/Liberty Island`

---